# WHEN NATURE STRIKES
## 03 – Temporal Trends Analysis

### 1. Introduction

This analysis examines how natural disaster occurrence
and their impacts have changed over time.

The objective is to identify long-term patterns,
compare disaster frequency across decades and investigate
changes in human and economic consequences.

### 2. Research Questions

1. How has natural disaster frequency changed over time?
2. Which decades recorded the highest disaster frequency?
3. How have reported deaths changed across decades?
4. How has the affected population changed over time?
5. How have inflation-adjusted economic losses evolved?
6. Are there differences between historical and recent records?

### 3. Dataset

- Primary dataset: natural_disaster_events.csv
- Source: EM-DAT, CRED / UCLouvain
- Time coverage: 1900–2026
- Total unique events: 15,727

### 4. Methodology

- Group unique disaster events by Start Year.
- Calculate annual disaster frequency.
- Group events by Decade.
- Calculate decadal frequency and reported impacts.
- Compare deaths, affected population and adjusted damage.
- Examine reporting coverage across time periods.

### 5. Important Considerations

- Each event is counted only once.
- Missing impact values are not treated as zero.
- Economic damage uses inflation-adjusted million US dollars.
- Historical records before 2000 have lower reporting quality.
- The year 2026 is incomplete and should be interpreted cautiously.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

print("Libraries imported successfully!")

In [ ]:
ROOT = Path.cwd()

if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

events = pd.read_csv(
    DATA_DIR / "natural_disaster_events.csv",
    encoding="utf-8-sig"
)

print("Dataset loaded successfully!")
print("Total events:", len(events))

In [ ]:
# Convert year and decade columns to numeric

events["Start Year"] = pd.to_numeric(
    events["Start Year"],
    errors="coerce"
)

events["Decade"] = pd.to_numeric(
    events["Decade"],
    errors="coerce"
)

# Create a clean analysis dataset

time_data = events.dropna(
    subset=["Start Year", "Decade"]
).copy()

time_data["Start Year"] = time_data["Start Year"].astype(int)
time_data["Decade"] = time_data["Decade"].astype(int)

print("Earliest year:", time_data["Start Year"].min())
print("Latest year:", time_data["Start Year"].max())
print("Records available:", len(time_data))

In [ ]:
#yearly disaster frequency

In [ ]:
yearly_frequency = (
    time_data.groupby("Start Year")["Event ID"]
    .nunique()
    .reset_index(name="Unique Events")
)

display(yearly_frequency.head(10))

In [ ]:
plt.figure(figsize=(15, 7))

sns.lineplot(
    data=yearly_frequency,
    x="Start Year",
    y="Unique Events",
    linewidth=1.8
)

plt.title(
    "Yearly Frequency of Natural Disasters (1900–2026)",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Year")
plt.ylabel("Number of Unique Disaster Events")

plt.tight_layout()

trend_dir = ROOT / "visualizations" / "trends"
trend_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    trend_dir / "01_yearly_disaster_frequency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#disaster frequency by decade

In [ ]:
decadal_frequency = (
    time_data.groupby("Decade")["Event ID"]
    .nunique()
    .reset_index(name="Unique Events")
)

display(decadal_frequency)

In [ ]:
plt.figure(figsize=(14, 7))

ax = sns.barplot(
    data=decadal_frequency,
    x="Decade",
    y="Unique Events",
    color="steelblue"
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

plt.title(
    "Natural Disaster Frequency by Decade",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Decade")
plt.ylabel("Number of Unique Events")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    trend_dir / "02_decadal_disaster_frequency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
top_decades = decadal_frequency.sort_values(
    "Unique Events",
    ascending=False
)

display(top_decades.head(5))

print(
    "Decade with highest recorded frequency:",
    top_decades.iloc[0]["Decade"]
)

In [ ]:
#human impacts across decades

In [ ]:
impact_columns = [
    "Total Deaths",
    "Total Affected",
    "Total Damage, Adjusted (million US$)"
]

decadal_impact = (
    time_data.groupby("Decade")
    .agg(
        Unique_Events=("Event ID", "nunique"),
        **{
            "Reported Death Events": (
                "Total Deaths", "count"
            ),
            "Total Deaths": (
                "Total Deaths",
                lambda x: x.sum(min_count=1)
            ),
            "Average Deaths": (
                "Total Deaths", "mean"
            ),
            "Reported Affected Events": (
                "Total Affected", "count"
            ),
            "Total Affected": (
                "Total Affected",
                lambda x: x.sum(min_count=1)
            ),
            "Average Affected": (
                "Total Affected", "mean"
            ),
            "Reported Damage Events": (
                "Total Damage, Adjusted (million US$)",
                "count"
            ),
            "Total Adjusted Damage": (
                "Total Damage, Adjusted (million US$)",
                lambda x: x.sum(min_count=1)
            ),
            "Average Adjusted Damage": (
                "Total Damage, Adjusted (million US$)",
                "mean"
            )
        }
    )
    .reset_index()
)

display(decadal_impact)

In [ ]:
death_data = decadal_impact.dropna(
    subset=["Total Deaths"]
)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=death_data,
    x="Decade",
    y="Total Deaths",
    color="firebrick"
)

plt.title(
    "Total Reported Natural Disaster Deaths by Decade",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Decade")
plt.ylabel("Total Reported Deaths")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    trend_dir / "03_decadal_deaths.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
affected_data = decadal_impact.dropna(
    subset=["Total Affected"]
)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=affected_data,
    x="Decade",
    y="Total Affected",
    color="darkorange"
)

plt.title(
    "Total Reported Population Affected by Decade",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Decade")
plt.ylabel("Total Affected Population")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    trend_dir / "04_decadal_affected_population.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#economic damage across decades

In [ ]:
damage_data = decadal_impact.dropna(
    subset=["Total Adjusted Damage"]
)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=damage_data,
    x="Decade",
    y="Total Adjusted Damage",
    color="seagreen"
)

plt.title(
    "Total Inflation-Adjusted Economic Damage by Decade",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Decade")
plt.ylabel("Adjusted Damage (Million US$)")
plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    trend_dir / "05_decadal_economic_damage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
decadal_impact["Death Coverage (%)"] = (
    decadal_impact["Reported Death Events"]
    / decadal_impact["Unique_Events"] * 100
)

decadal_impact["Affected Coverage (%)"] = (
    decadal_impact["Reported Affected Events"]
    / decadal_impact["Unique_Events"] * 100
)

decadal_impact["Damage Coverage (%)"] = (
    decadal_impact["Reported Damage Events"]
    / decadal_impact["Unique_Events"] * 100
)

display(
    decadal_impact[
        [
            "Decade",
            "Death Coverage (%)",
            "Affected Coverage (%)",
            "Damage Coverage (%)"
        ]
    ].round(2)
)

In [ ]:
coverage_columns = [
    "Death Coverage (%)",
    "Affected Coverage (%)",
    "Damage Coverage (%)"
]

plt.figure(figsize=(14, 7))

for column in coverage_columns:
    plt.plot(
        decadal_impact["Decade"],
        decadal_impact[column],
        marker="o",
        label=column
    )

plt.title(
    "Impact Reporting Coverage Across Decades",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Decade")
plt.ylabel("Reported Events (%)")
plt.ylim(0, 105)
plt.legend()

plt.tight_layout()

plt.savefig(
    trend_dir / "06_decadal_reporting_coverage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#historical and recent periods

In [ ]:
time_data["Period"] = np.where(
    time_data["Start Year"] < 2000,
    "Before 2000",
    "2000 Onwards"
)

period_comparison = (
    time_data.groupby("Period")
    .agg(
        Unique_Events=("Event ID", "nunique"),
        Reported_Deaths=("Total Deaths", "count"),
        Total_Deaths=("Total Deaths",
                      lambda x: x.sum(min_count=1)),
        Reported_Affected=("Total Affected", "count"),
        Total_Affected=("Total Affected",
                        lambda x: x.sum(min_count=1)),
        Reported_Damage=(
            "Total Damage, Adjusted (million US$)", "count"
        ),
        Total_Damage=(
            "Total Damage, Adjusted (million US$)",
            lambda x: x.sum(min_count=1)
        )
    )
    .reset_index()
)

display(period_comparison)

In [ ]:
plt.figure(figsize=(9, 6))

sns.barplot(
    data=period_comparison,
    x="Period",
    y="Unique_Events",
    color="steelblue"
)

plt.title(
    "Natural Disaster Events: Historical vs Recent Period",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Time Period")
plt.ylabel("Unique Events")

plt.tight_layout()

plt.savefig(
    trend_dir / "07_historical_vs_recent.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
print("TEMPORAL ANALYSIS SUMMARY")

print(
    "\nHighest frequency year:",
    yearly_frequency.loc[
        yearly_frequency["Unique Events"].idxmax(),
        "Start Year"
    ]
)

print(
    "Maximum events in one year:",
    yearly_frequency["Unique Events"].max()
)

print(
    "\nHighest frequency decade:",
    top_decades.iloc[0]["Decade"]
)

print(
    "Maximum events in one decade:",
    top_decades.iloc[0]["Unique Events"]
)

print("\nDECADAL IMPACT SUMMARY")

display(
    decadal_impact[
        [
            "Decade",
            "Unique_Events",
            "Total Deaths",
            "Total Affected",
            "Total Adjusted Damage",
            "Death Coverage (%)",
            "Affected Coverage (%)",
            "Damage Coverage (%)"
        ]
    ].round(2)
)

In [ ]:
output_dir = ROOT / "analysis" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

yearly_frequency.to_csv(
    output_dir / "yearly_disaster_frequency.csv",
    index=False
)

decadal_frequency.to_csv(
    output_dir / "decadal_disaster_frequency.csv",
    index=False
)

decadal_impact.to_csv(
    output_dir / "decadal_disaster_impact.csv",
    index=False
)

period_comparison.to_csv(
    output_dir / "historical_recent_comparison.csv",
    index=False
)

print("Temporal analysis results exported successfully!")